In [15]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv("C:/Users/gdani/OneDrive/Desktop/master thesis/datasets/HDMA/X_df_selected.csv", delimiter=",")
df

,census_tract,manufactured_home_secured_property_type,rate_spread,income,open-end_line_of_credit,interest_rate,loan_amount,applicant_credit_score_type,derived_msa-md,tract_minority_population_percent,...,denial_reason-1,tract_population,derived_dwelling_category_Single Family (1-4 Units):Manufactured,derived_dwelling_category_Single Family (1-4 Units):Site-Built,origination_charges_0.0,discount_points_Exempt,lender_credits_Exempt,debt_to_income_ratio_>60%,target,applicant_sex
0,0.272063,-0.084555,0.002435,-0.043407,-0.082837,3.162580,0.294451,-0.029675,-0.788599,0.590077,...,-0.061467,-0.911072,0.0,1.0,1.0,1.0,1.0,0.0,1,4
1,-0.422953,-0.084555,0.002435,-0.043407,-0.082837,1.225837,-0.028293,-0.029675,-0.513287,1.110555,...,-0.061467,-1.139734,0.0,1.0,1.0,1.0,1.0,0.0,1,4
2,-0.971887,-0.084555,0.002435,-0.043407,-0.082837,3.700565,-0.125117,-0.029675,-0.599496,1.577268,...,-0.061467,0.859477,0.0,1.0,1.0,1.0,1.0,0.0,1,4
3,-0.222835,-0.084555,0.002435,-0.043407,-0.082837,0.687853,-0.168149,-0.029675,2.105616,1.914766,...,-0.061467,-0.457633,0.0,1.0,1.0,1.0,1.0,0.0,1,4
4,-1.556956,-0.084555,0.002435,-0.043407,-0.082837,2.983252,0.025497,-0.029675,-0.160109,-1.122711,...,-0.061467,-1.367424,0.0,1.0,1.0,1.0,1.0,0.0,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
326609,0.455133,-0.084555,0.002435,0.017484,-0.093409,-0.036633,-0.049810,-0.103982,0.329334,-1.027380,...,-0.129342,-0.083813,0.0,1.0,1.0,1.0,1.0,1.0,0,1
326610,-1.227684,-0.084555,0.533711,0.114193,-0.082837,1.476896,-0.103601,-0.103982,-0.657895,-0.870605,...,-0.061467,0.654119,0.0,1.0,1.0,1.0,1.0,0.0,1,1
326611,1.589101,-0.084555,-0.073461,-0.045795,-0.082837,0.239532,-0.039052,-0.029675,-0.531363,-0.359615,...,-0.061467,0.159413,0.0,1.0,0.0,0.0,1.0,0.0,1,3
326612,1.277749,-0.084555,-0.299100,-0.091164,-0.082837,-0.477780,-0.049810,-0.093366,2.105616,0.161767,...,-0.061467,-2.296150,0.0,1.0,0.0,1.0,1.0,0.0,1,1


In [21]:
df['applicant_sex'] = df['applicant_sex'].apply(lambda x: 1 if x in [1, 3] else 2)
print(df['applicant_sex'].unique())

[2 1]


In [22]:
#let's start the data analysis
# Identify non-numeric columns
non_numeric = df.select_dtypes(exclude=['number'])

# Count them
num_non_numeric = non_numeric.shape[1]

print(f"Number of non-numeric variables: {num_non_numeric}")

Number of non-numeric variables: 0


In [23]:
#drop missing values
df = df.dropna()
#Check
print("\n",df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 326614 entries, 0 to 326613
Data columns (total 32 columns):
 #   Column                                                            Non-Null Count   Dtype  
---  ------                                                            --------------   -----  
 0   census_tract                                                      326614 non-null  float64
 1   manufactured_home_secured_property_type                           326614 non-null  float64
 2   rate_spread                                                       326614 non-null  float64
 3   income                                                            326614 non-null  float64
 4   open-end_line_of_credit                                           326614 non-null  float64
 5   interest_rate                                                     326614 non-null  float64
 6   loan_amount                                                       326614 non-null  float64
 7   applicant_credit_sco

In [25]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import aif360
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing

# Define label and protected attribute
label_col = 'target'
protected_attr = 'applicant_sex'  # 1 = male (privileged), 2 = female (unprivileged)

# Split into features and target
X = df.drop(columns=[label_col])
y = df[label_col]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train base classifier
clf = LogisticRegression(solver='saga', max_iter=1000)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)

# Prepare AIF360 datasets
aif_test = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    #
    df=X_test.assign(target=y_test.values),
    label_names=[label_col],
    protected_attribute_names=[protected_attr]
)

aif_pred = aif_test.copy()
aif_pred.labels = y_pred.reshape(-1, 1)

# Apply Calibrated Equalized Odds Postprocessing
eqodds = CalibratedEqOddsPostprocessing(
    privileged_groups=[{protected_attr: 1}],
    unprivileged_groups=[{protected_attr: 2}],
    cost_constraint="weighted"
)
eqodds = eqodds.fit(aif_test, aif_pred)
aif_eqodds_pred = eqodds.predict(aif_pred)

# Evaluate before fairness
metric_orig = ClassificationMetric(
    aif_test, aif_pred,
    privileged_groups=[{protected_attr: 1}],
    unprivileged_groups=[{protected_attr: 2}]
)

# Evaluate after fairness
metric_post = ClassificationMetric(
    aif_test, aif_eqodds_pred,
    privileged_groups=[{protected_attr: 1}],
    unprivileged_groups=[{protected_attr: 2}]
)

# Print results
print("=== BEFORE Fairness Postprocessing ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Equal Opportunity Difference:", metric_orig.equal_opportunity_difference())
print("Equalized Odds Difference:", metric_orig.average_odds_difference())
print("Disparate impact:", metric_orig.disparate_impact())

print("\n=== AFTER Calibrated Equalized Odds ===")
print("Accuracy:", accuracy_score(y_test, aif_eqodds_pred.labels))
print("Equal Opportunity Difference:", metric_post.equal_opportunity_difference())
print("Equalized Odds Difference:", metric_post.average_odds_difference())
print("Disparate impact:", metric_post.disparate_impact())

C:\Users\gdani\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


=== BEFORE Fairness Postprocessing ===
Accuracy: 0.9660303415336099
Equal Opportunity Difference: 0.00456252326436668
Equalized Odds Difference: 0.0030715676670187517
Disparate impact: 1.026378002989289

=== AFTER Calibrated Equalized Odds ===
Accuracy: 1.0
Equal Opportunity Difference: 0.0
Equalized Odds Difference: 0.0
Disparate impact: 1.021187701202311


In [26]:
print("Protected attribute values:", np.unique(aif_test.protected_attributes, return_counts=True))
print("Label values:", np.unique(aif_test.labels, return_counts=True))
print("Predicted label values:", np.unique(aif_pred.labels, return_counts=True))


Protected attribute values: (array([1., 2.]), array([37699, 27624]))
Label values: (array([0., 1.]), array([16623, 48700]))
Predicted label values: (array([0, 1]), array([18574, 46749]))


In [27]:
from sklearn.metrics import confusion_matrix

# BEFORE fairness
y_true_before = aif_test.labels.ravel()
y_pred_before = aif_pred.labels.ravel()

# AFTER fairness
y_pred_after = aif_eqodds_pred.labels.ravel()

# Confusion matrices
cm_before = confusion_matrix(y_true_before, y_pred_before)
cm_after = confusion_matrix(y_true_before, y_pred_after)

# Unpack counts (assuming binary classification with labels 0 and 1)
tn_b, fp_b, fn_b, tp_b = cm_before.ravel()
tn_a, fp_a, fn_a, tp_a = cm_after.ravel()

# Print BEFORE fairness
print("=== BEFORE Fairness ===")
print(f"True Positives:  {tp_b}")
print(f"False Positives: {fp_b}")
print(f"True Negatives:  {tn_b}")
print(f"False Negatives: {fn_b}")

# Print AFTER fairness
print("\n=== AFTER Fairness ===")
print(f"True Positives:  {tp_a}")
print(f"False Positives: {fp_a}")
print(f"True Negatives:  {tn_a}")
print(f"False Negatives: {fn_a}")


=== BEFORE Fairness ===
True Positives:  46615
False Positives: 134
True Negatives:  16489
False Negatives: 2085

=== AFTER Fairness ===
True Positives:  48700
False Positives: 0
True Negatives:  16623
False Negatives: 0
